In [ ]:
# Week 2 - Day 1: Feature Engineering

## Project: Customer Churn Prediction & Lifetime Value (LTV)

### Objective
Create meaningful features from customer tenure, monthly charges,
and total charges to support predictive churn modeling.

In [3]:
import pandas as pd
import numpy as np

# Load Telco Customer Churn dataset
df = pd.read_csv("01_Dataset_Telco-Customer-Churn.csv")

# Display first 5 rows
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
# Check dataset structure
df.shape

(7043, 21)

In [5]:
# Check data types and missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [6]:
# Check missing values
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [7]:
# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Check missing values after conversion
df["TotalCharges"].isnull().sum()

11

In [8]:
# Check the rows where TotalCharges is missing
df[df["TotalCharges"].isnull()][["customerID", "tenure", "MonthlyCharges", "TotalCharges"]]

,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,NaN
753,3115-CZMZD,0,20.25,NaN
936,5709-LVOEQ,0,80.85,NaN
1082,4367-NUYAO,0,25.75,NaN
1340,1371-DWPAZ,0,56.05,NaN
3331,7644-OMVMY,0,19.85,NaN
3826,3213-VVOLG,0,25.35,NaN
4380,2520-SGTTA,0,20.00,NaN
5218,2923-ARZLG,0,19.70,NaN
6670,4075-WKNIU,0,73.35,NaN


In [9]:
# Fill missing TotalCharges with 0 for customers with tenure = 0
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# Verify missing values
df["TotalCharges"].isnull().sum()

0

In [10]:
# Create TotalCharges per Tenure feature
df["TotalChargesPerTenure"] = np.where(
    df["tenure"] > 0,
    df["TotalCharges"] / df["tenure"],
    0
)

# Check the new feature
df[["tenure", "MonthlyCharges", "TotalCharges", "TotalChargesPerTenure"]].head()

,tenure,MonthlyCharges,TotalCharges,TotalChargesPerTenure
0,1,29.85,29.85,29.850000
1,34,56.95,1889.50,55.573529
2,2,53.85,108.15,54.075000
3,45,42.30,1840.75,40.905556
4,2,70.70,151.65,75.825000


In [11]:
# Create customer tenure groups
df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 24, 48, 60, 72],
    labels=["0-1 Year", "1-2 Years", "2-4 Years", "4-5 Years", "5-6 Years"]
)

# Check tenure groups
df["TenureGroup"].value_counts().sort_index()

TenureGroup
0-1 Year     2186
1-2 Years    1024
2-4 Years    1594
4-5 Years     832
5-6 Years    1407
Name: count, dtype: int64

In [12]:
# Create monthly charge to tenure ratio
df["MonthlyChargePerTenure"] = np.where(
    df["tenure"] > 0,
    df["MonthlyCharges"] / df["tenure"],
    0
)

# Check the new feature
df[["tenure", "MonthlyCharges", "MonthlyChargePerTenure"]].head()

,tenure,MonthlyCharges,MonthlyChargePerTenure
0,1,29.85,29.850
1,34,56.95,1.675
2,2,53.85,26.925
3,45,42.30,0.940
4,2,70.70,35.350


In [13]:
# Check the distribution of the new feature
df["MonthlyChargePerTenure"].describe()

count    7043.000000
mean        8.606624
std        16.342826
min         0.000000
25%         1.275176
50%         2.141892
75%         6.600000
max       102.450000
Name: MonthlyChargePerTenure, dtype: float64

In [16]:
# Convert Churn into numeric values
df["ChurnNumeric"] = df["Churn"].map({"Yes": 1, "No": 0})

# Check correlation with the new feature
df[["MonthlyChargePerTenure", "ChurnNumeric"]].corr()

,MonthlyChargePerTenure,ChurnNumeric
MonthlyChargePerTenure,1.000000,0.386367
ChurnNumeric,0.386367,1.000000


In [17]:
# Create a summary of the new features
df[[
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "MonthlyChargePerTenure",
    "TotalChargesPerTenure",
    "TenureGroup"
]].head(10)

,tenure,MonthlyCharges,TotalCharges,MonthlyChargePerTenure,TotalChargesPerTenure,TenureGroup
0,1,29.85,29.85,29.850000,29.850000,0-1 Year
1,34,56.95,1889.50,1.675000,55.573529,2-4 Years
2,2,53.85,108.15,26.925000,54.075000,0-1 Year
3,45,42.30,1840.75,0.940000,40.905556,2-4 Years
4,2,70.70,151.65,35.350000,75.825000,0-1 Year
5,8,99.65,820.50,12.456250,102.562500,0-1 Year
6,22,89.10,1949.40,4.050000,88.609091,1-2 Years
7,10,29.75,301.90,2.975000,30.190000,0-1 Year
8,28,104.80,3046.05,3.742857,108.787500,2-4 Years
9,62,56.15,3487.95,0.905645,56.257258,5-6 Years


In [18]:
# Save the feature-engineered dataset
df.to_csv("telco_customer_churn_feature_engineered.csv", index=False)

print("Feature-engineered dataset saved successfully.")

Feature-engineered dataset saved successfully.
